# Legal Extraction on Kaggle

This notebook is the project-facing Kaggle path for `004-llm-legal-extraction`.

It is intentionally self-contained for Kaggle's single-session workflow. The notebook:
1. clones the project branch
2. stages a prebuilt `llama-server` runtime artifact into `/kaggle/working`
3. starts the server in the same session
4. loads the legal preview corpus
5. calls the local LLM with the `004` proposition contract
6. runs `run_legal_semantic_validation_cycle()`
7. exports extraction and validation artifacts

This notebook does not build or publish `llama-server`. It assumes a prebuilt runtime dataset is attached.


In [ ]:
# RUN GUARD
RUN_NOTEBOOK = False

# REQUIRED OPERATOR INPUTS
REPO_URL = "https://github.com/lexandree/chat-bot.git"
REPO_BRANCH = "004-llm-legal-extraction"
REPO_ROOT = "/kaggle/working/chat-bot"
PREVIEW_PATH = "/kaggle/input/path-to-your-preview/legal_xml_import_preview.json"
LLAMA_SERVER_ARTIFACT_ROOT = "/kaggle/input/path-to-your-llama-server-runtime"
MODEL_FILE = "/kaggle/input/path-to-your-model/model.gguf"
MMPROJ_FILE = ""

# OPTIONAL OPERATOR INPUTS
EXPORT_NAME = None
MAX_DOCUMENTS = 8
LAW_CODES = []  # Example: ["AufenthG"]
MAX_NEW_TOKENS = 512
MODEL_ID_OVERRIDE = None
TEMPERATURE = 0.0
HTTP_TIMEOUT_SECONDS = 120
STARTUP_TIMEOUT_SECONDS = 120
RUNTIME_CONTOUR = "operator_managed"
RUN_NAME = "kaggle-legal-semantic-validation"
VALIDATION_FIXTURE_PATH = "tests/fixtures/legal_extraction_validation_cases.json"
EXPORT_ROOT = "/kaggle/working/legal_extraction_artifacts"
LEGAL_MANAGED_SMOKE_LIMIT = 5
MAX_STABLE_CONTEXT_OBSERVED = None
PROMPT_OR_POLICY_VERSION = "legal_extraction_v1"

LLAMA_SERVER_HOST = "127.0.0.1"
LLAMA_SERVER_PORT = 18081
CTX_SIZE = 8192
BATCH_SIZE = 8192
UBATCH_SIZE = 512
PARALLEL = 1
N_GPU_LAYERS = "all"
FLASH_ATTN = "on"
SPLIT_MODE = None
TENSOR_SPLIT = None
SLOTS_ENDPOINT = None
EXTRA_SERVER_ARGS = [
    "--cache-type-k", "q4_0",
    "--cache-type-v", "q4_0",
    "--reasoning", "off",
    "--reasoning-budget", "0",
    "--reasoning-format", "none",
]
ENV_OVERRIDES = {
    "LEGAL_RUNTIME_CONTOUR": RUNTIME_CONTOUR,
    "LEGAL_MANAGED_SMOKE_LIMIT": str(LEGAL_MANAGED_SMOKE_LIMIT),
}

if not RUN_NOTEBOOK:
    raise RuntimeError(
        "Notebook execution is disabled. Set RUN_NOTEBOOK = True only after confirming the preview path, runtime artifact path, model path, and environment variables."
    )


In [ ]:
# Project bootstrap stays in one cell because clone, install, and env preparation
# form one setup boundary. Running them together reduces partial-state confusion.

import json
import os
import shutil
import signal
import socket
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import requests

repo_root = Path(REPO_ROOT)
if repo_root.exists():
    subprocess.run(["git", "-C", str(repo_root), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_root), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(repo_root)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_root)], check=True)

for key, value in ENV_OVERRIDES.items():
    os.environ[key] = value

if str(repo_root / "src") not in sys.path:
    sys.path.append(str(repo_root / "src"))

artifact_root = Path(LLAMA_SERVER_ARTIFACT_ROOT)
if not artifact_root.exists():
    raise FileNotFoundError(f"LLAMA_SERVER_ARTIFACT_ROOT does not exist: {artifact_root}")

model_file = Path(MODEL_FILE)
if not model_file.exists() or not model_file.is_file():
    raise FileNotFoundError(f"MODEL_FILE does not exist: {model_file}")

mmproj_file = Path(MMPROJ_FILE) if MMPROJ_FILE else None
if mmproj_file is not None and (not mmproj_file.exists() or not mmproj_file.is_file()):
    raise FileNotFoundError(f"MMPROJ_FILE does not exist: {mmproj_file}")

work_root = Path("/kaggle/working/legal_extraction_runtime")
runtime_root = work_root / "llama_server"
log_dir = work_root / "logs"
run_dir = work_root / "run"
for root in (work_root, runtime_root, log_dir, run_dir):
    root.mkdir(parents=True, exist_ok=True)

pid_file = run_dir / "llama-server.pid"
cmd_file = run_dir / "llama-server.command.json"
log_file = log_dir / "llama-server.log"
llama_server_base_url = f"http://{LLAMA_SERVER_HOST}:{LLAMA_SERVER_PORT}"

print("REPO_ROOT =", repo_root)
print("PREVIEW_PATH =", PREVIEW_PATH)
print("LLAMA_SERVER_ARTIFACT_ROOT =", artifact_root)
print("MODEL_FILE =", model_file)
print("LLAMA_SERVER_BASE_URL =", llama_server_base_url)


In [ ]:
# Runtime helpers stay together because staging, launch, health checks, and cleanup
# all share the same pid file, log file, and runtime directory assumptions.

from bot.main import build_application, run_legal_semantic_validation_cycle
from ingestion.legal_proposition_extractor import build_proposition_prompt, parse_proposition_output


def stage_runtime_artifact(dataset_root: Path, out_root: Path) -> Path:
    """Copy a supported prebuilt llama-server runtime layout into working storage."""
    if out_root.exists():
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if (dataset_root / "llama-server").exists():
        source_root = dataset_root
    elif (dataset_root / "runtime" / "llama-server").exists():
        source_root = dataset_root / "runtime"
    elif (dataset_root / "bin" / "llama-server").exists():
        source_root = dataset_root / "bin"
    else:
        raise FileNotFoundError(f"No supported llama-server layout found under {dataset_root}")
    for item in source_root.iterdir():
        if item.is_file():
            shutil.copy2(item, out_root / item.name)
    return out_root


def clean_env(runtime_dir: Path) -> dict[str, str]:
    """Build a minimal launch environment for the staged llama-server runtime."""
    env = {
        "HOME": os.environ.get("HOME", "/kaggle/working"),
        "PATH": "/usr/bin:/bin",
        "LANG": "C.UTF-8",
        "LC_ALL": "C.UTF-8",
    }
    ld_parts = [str(runtime_dir)]
    if os.environ.get("LD_LIBRARY_PATH"):
        ld_parts.append(os.environ["LD_LIBRARY_PATH"])
    env["LD_LIBRARY_PATH"] = ":".join(ld_parts)
    return env


def read_pid() -> int | None:
    """Read the current server pid from the pid file when available."""
    if pid_file.exists():
        try:
            return int(pid_file.read_text().strip())
        except Exception:
            return None
    return None


def process_state(pid: int) -> str | None:
    """Return the one-letter /proc state code for the given pid when available."""
    status_path = Path(f"/proc/{pid}/status")
    if not status_path.exists():
        return None
    for line in status_path.read_text(errors="ignore").splitlines():
        if line.startswith("State:"):
            raw = line.split(":", 1)[1].strip()
            return raw.split()[0] if raw else None
    return None


def process_exists(pid: int) -> bool:
    """Return True when the pid exists and is not already a zombie."""
    if process_state(pid) == "Z":
        return False
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def wait_for_pid_exit(pid: int, timeout_s: float = 20.0, poll_s: float = 0.5) -> bool:
    """Poll until a tracked process exits or the timeout expires."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if not process_exists(pid):
            return True
        time.sleep(poll_s)
    return not process_exists(pid)


def tail_log(n: int = 120) -> str:
    """Return the last log lines for notebook-side diagnostics."""
    if not log_file.exists():
        return ""
    lines = log_file.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])


def is_port_open(host: str | None = None, port: int | None = None, timeout: float = 1.0) -> bool:
    """Return True when the host and port accept a TCP connection."""
    host = LLAMA_SERVER_HOST if host is None else host
    port = LLAMA_SERVER_PORT if port is None else port
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def build_server_command(runtime_dir: Path, model_path: Path) -> list[str]:
    """Build the llama-server launch command for the staged runtime."""
    wrapper = runtime_dir / "llama-server.sh"
    binary = wrapper if wrapper.exists() else (runtime_dir / "llama-server")
    cmd = [
        str(binary),
        "-m", str(model_path),
        "--host", str(LLAMA_SERVER_HOST),
        "--port", str(LLAMA_SERVER_PORT),
        "--ctx-size", str(CTX_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--ubatch-size", str(UBATCH_SIZE),
        "--parallel", str(PARALLEL),
        "--n-gpu-layers", str(N_GPU_LAYERS),
        "--flash-attn", str(FLASH_ATTN),
    ]
    if SPLIT_MODE is not None:
        cmd += ["--split-mode", str(SPLIT_MODE)]
    if TENSOR_SPLIT is not None:
        cmd += ["--tensor-split", str(TENSOR_SPLIT)]
    if SLOTS_ENDPOINT is True:
        cmd.append("--slots")
    elif SLOTS_ENDPOINT is False:
        cmd.append("--no-slots")
    if mmproj_file is not None:
        cmd += ["--mmproj", str(mmproj_file)]
    cmd.extend(str(item) for item in EXTRA_SERVER_ARGS)
    return cmd


def stop_server(force: bool = False, timeout_s: float = 20.0) -> bool:
    """Stop the tracked llama-server process and clean stale or zombie pid files."""
    pid = read_pid()
    if pid is None:
        pid_file.unlink(missing_ok=True)
        return True
    if process_state(pid) == "Z":
        pid_file.unlink(missing_ok=True)
        return True
    if not process_exists(pid):
        pid_file.unlink(missing_ok=True)
        return True
    os.kill(pid, signal.SIGKILL if force else signal.SIGTERM)
    stopped = wait_for_pid_exit(pid, timeout_s=timeout_s)
    if stopped:
        pid_file.unlink(missing_ok=True)
        return True
    if force:
        raise RuntimeError(f"PID {pid} is still alive even after SIGKILL.")
    raise RuntimeError(f"PID {pid} did not exit after SIGTERM. Call stop_server(force=True) only if you explicitly want SIGKILL.")


def wait_for_server(timeout_s: int | None = None) -> bool:
    """Poll the health endpoint until llama-server becomes ready."""
    timeout_s = STARTUP_TIMEOUT_SECONDS if timeout_s is None else timeout_s
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if log_file.exists():
            tail = tail_log(80)
            if "failed to load model" in tail or "error loading model" in tail:
                return False
        try:
            response = requests.get(f"{llama_server_base_url}/health", timeout=3)
            if response.status_code < 500:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False


def start_server() -> subprocess.Popen:
    """Stage the runtime artifact, launch llama-server, and record pid and command metadata."""
    stop_server(force=False)
    stage_runtime_artifact(artifact_root, runtime_root)
    cmd = build_server_command(runtime_root, model_file)
    env = clean_env(runtime_root)
    with open(log_file, "w") as logf:
        proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
    pid_file.write_text(str(proc.pid), encoding="utf-8")
    cmd_file.write_text(json.dumps(cmd, indent=2, ensure_ascii=False), encoding="utf-8")
    return proc


def server_health() -> dict[str, Any]:
    """Return a notebook-friendly health response object from the local runtime."""
    try:
        response = requests.get(f"{llama_server_base_url}/health", timeout=5)
        return {"status_code": response.status_code, "text": response.text}
    except Exception as exc:
        return {"error": str(exc)}


def server_models() -> Any:
    """Return model metadata from the local runtime when exposed."""
    for path in ("/v1/models", "/models"):
        try:
            response = requests.get(f"{llama_server_base_url}{path}", timeout=10)
            if response.ok:
                return response.json()
        except Exception:
            pass
    return None


def load_preview_documents(preview_path: str, max_documents: int | None = None, law_codes: list[str] | None = None) -> list[dict[str, Any]]:
    """Load normalized legal documents from either top-level or imported-source preview JSON."""
    payload = json.loads(Path(preview_path).read_text(encoding="utf-8"))
    documents = list(payload.get("documents", []))
    if not documents:
        for imported_source in payload.get("imported_sources", []):
            documents.extend(imported_source.get("documents", []))
    if law_codes:
        allowed = set(law_codes)
        documents = [doc for doc in documents if doc.get("law_code") in allowed]
    if max_documents is not None:
        documents = documents[:max_documents]
    return documents


def extract_json_object(text: str) -> dict[str, Any]:
    """Extract the first top-level JSON object from model output."""
    text = text.strip()
    if text.startswith("{"):
        return json.loads(text)
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end + 1])


def resolve_model_id(model_id_override: str | None = None) -> str:
    """Resolve the active model id from llama-server unless the operator forces one."""
    if model_id_override:
        return model_id_override
    payload = server_models()
    if isinstance(payload, dict) and payload.get("data"):
        return payload["data"][0]["id"]
    raise RuntimeError("Could not resolve an active model id from llama-server. Set MODEL_ID_OVERRIDE explicitly.")


def extract_message_text(message: Any) -> str:
    """Normalize visible assistant text from common OpenAI-compatible response shapes."""
    if isinstance(message, dict):
        content = message.get("content")
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            parts: list[str] = []
            for item in content:
                if isinstance(item, dict) and isinstance(item.get("text"), str):
                    parts.append(item["text"])
            return "".join(parts)
    return str(message)


def call_llama_server(prompt: str, *, model_id: str) -> str:
    """Send one non-streaming chat completion request to the local llama-server runtime."""
    payload = {
        "model": model_id,
        "messages": [
            {
                "role": "system",
                "content": "You extract reviewable legal propositions from statutory text. Return JSON only. Do not output markdown or explanations.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_NEW_TOKENS,
        "stream": False,
    }
    response = requests.post(
        f"{llama_server_base_url}/v1/chat/completions",
        json=payload,
        timeout=HTTP_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()
    return extract_message_text(payload["choices"][0]["message"])


def build_notebook_extractor(*, model_id: str, prompt_or_policy_version: str):
    """Create the `004` extractor callable expected by `run_legal_semantic_validation_cycle()`."""
    def extractor(**kwargs: Any) -> dict[str, Any]:
        prompt = build_proposition_prompt(
            fragment=kwargs["fragment"],
            section=kwargs["section"],
            prompt_or_policy_version=prompt_or_policy_version,
        )
        raw_text = call_llama_server(prompt, model_id=model_id)
        payload = extract_json_object(raw_text)
        propositions = parse_proposition_output(payload)
        return {"propositions": propositions}
    return extractor


In [ ]:
# Launch and health checks stay in one cell because server state is the key transition
# between setup and extraction. This is the right place to fail early on runtime issues.

proc = start_server()
print("Started PID:", proc.pid)
ready = wait_for_server()
print("Server ready:", ready)
print("Port open:", is_port_open())
print("Health:", json.dumps(server_health(), indent=2, ensure_ascii=False))
models_payload = server_models()
print("Models:", json.dumps(models_payload, indent=2, ensure_ascii=False)[:2000] if models_payload else None)
print("Log tail:")
print(tail_log(120))
if not ready:
    raise RuntimeError("llama-server did not become ready. Inspect the log tail above before continuing.")


In [ ]:
# This run cell keeps the actual `004` workflow short and auditable:
# load documents, resolve the model, build the extractor, run semantic validation,
# and leave the detailed artifact export to the next cell.

app = build_application()
documents = load_preview_documents(PREVIEW_PATH, max_documents=MAX_DOCUMENTS, law_codes=LAW_CODES or None)
model_id = resolve_model_id(MODEL_ID_OVERRIDE)
extractor = build_notebook_extractor(
    model_id=model_id,
    prompt_or_policy_version=PROMPT_OR_POLICY_VERSION,
)

started_at = time.time()
result = run_legal_semantic_validation_cycle(
    app,
    documents,
    extractor=extractor,
    fixture_path=VALIDATION_FIXTURE_PATH,
    run_name=RUN_NAME,
    runtime_contour=RUNTIME_CONTOUR,
)
elapsed_seconds = round(time.time() - started_at, 3)

summary = {
    "document_count": len(documents),
    "model_id": model_id,
    "elapsed_seconds": elapsed_seconds,
    "processed_count": result["extraction"]["run"]["processed_count"],
    "persisted_candidate_count": len(result["extraction"]["candidates"]),
    "validation_case_count": len(result["validation"]["cases"]),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# Export stays separate so failed runs do not overwrite previous artifacts.
# The directory can later be downloaded or versioned into a private Kaggle dataset.

export_id = EXPORT_NAME or time.strftime("%Y-%m-%dT%H-%M-%SZ_legal_extraction")
export_root = Path(EXPORT_ROOT) / export_id
export_root.mkdir(parents=True, exist_ok=True)

run_manifest = {
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_root": REPO_ROOT,
    "preview_path": PREVIEW_PATH,
    "validation_fixture_path": VALIDATION_FIXTURE_PATH,
    "llama_server_artifact_root": str(artifact_root),
    "llama_server_base_url": llama_server_base_url,
    "model_file": str(model_file),
    "mmproj_file": str(mmproj_file) if mmproj_file else None,
    "model_id": model_id,
    "runtime_contour": RUNTIME_CONTOUR,
    "run_name": RUN_NAME,
    "max_documents": MAX_DOCUMENTS,
    "law_codes": LAW_CODES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "http_timeout_seconds": HTTP_TIMEOUT_SECONDS,
    "startup_timeout_seconds": STARTUP_TIMEOUT_SECONDS,
    "prompt_or_policy_version": PROMPT_OR_POLICY_VERSION,
    "max_stable_context_observed": MAX_STABLE_CONTEXT_OBSERVED,
    "elapsed_seconds": elapsed_seconds,
    "ctx_size": CTX_SIZE,
    "batch_size": BATCH_SIZE,
    "ubatch_size": UBATCH_SIZE,
    "parallel": PARALLEL,
    "n_gpu_layers": N_GPU_LAYERS,
    "flash_attn": FLASH_ATTN,
    "split_mode": SPLIT_MODE,
    "tensor_split": TENSOR_SPLIT,
    "extra_server_args": EXTRA_SERVER_ARGS,
}

profile_report = {
    "summary": summary,
    "max_stable_context_observed": MAX_STABLE_CONTEXT_OBSERVED,
    "server_health": server_health(),
    "server_models": server_models(),
    "validation_result": result["validation"],
}

(export_root / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(export_root / "profile_report.json").write_text(json.dumps(profile_report, indent=2, ensure_ascii=False), encoding="utf-8")
(export_root / "semantic_validation_result.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
if log_file.exists():
    shutil.copy2(log_file, export_root / log_file.name)
if cmd_file.exists():
    shutil.copy2(cmd_file, export_root / cmd_file.name)

print("Artifacts written to", export_root)
print("Files:")
for item in sorted(export_root.iterdir()):
    print(" -", item.name)
